# Biblical Persona DPO Preference Data Generator (local vLLM, SFT adapter)

**Copy of `biblical_datagen_v2_dpo.ipynb` made 2026-11-10.** Two sets of changes from the original.

**Where it runs (first test run 2026-11-10, 234 pairs):**

- Calls a local vLLM stack on port 8005 instead of OpenRouter (`vllm-qwen38-27b-fp8-kvdisk` since 2026-11-11; the two test runs on 2026-11-10 used the NVFP4 stack). No API key, no spend. Works on the host or in the `unsloth-notebook` container.
- The rejected answers are written by `biblical_sft`, the Qwen3.8-27B SFT adapter that stack serves, with thinking off.
- `CONCURRENCY = 16`, matching the stack's `--max-num-seqs 16`.

**Recipe 2 (edited 2026-11-10 after that test run; not yet run, so the saved outputs below are still the first test run's):**

The first run's rejected answers were still too easy to tell from the chosen ones: voice drift read like a commentator, and platitudes were 62 words against 272 and in third person.

- **Every rejected answer is now written in persona.** The generation prompt starts with the persona's own identity line (the part of the trained prompt before `YOUR DISTINCTIVE VOICE`). The voice notes, example lines and rules are left out, and a strategy instruction is added. The pair itself still stores the full trained prompt.
- **Platitudes are full length and first person**, not "2-3 sentences, avoid first person".
- **Continuation tasks are no longer sampled** (the original took 403 of 3,564 pairs from them).
- **A judge step** (section 6b, base Qwen on the same stack) keeps only the pairs where the rejected answer really has the intended flaw. The original's gate checked format only.
- Writes to its own files (see the config cell). Neither the original 3,564 pairs nor earlier test runs' files are overwritten.

**Changes on 2026-11-11 (not yet run when written):** the `scripture_fabrication` strategy is switched off (`STRATEGIES` in the config cell; the SFT adapter almost never fabricates when it knows who it is), the generator is the SFT adapter retrained on 2026-11-10, the stack is the FP8 one, and output files are `biblical_personas_v4_dpo_local_sft*.jsonl`. Test runs write `_test` files and always start fresh.

Generate DPO (Direct Preference Optimization) training pairs from existing SFT data.

**What this does:** Takes the high-quality SFT training data (4,352 conversations across 26 biblical personas)
and generates chosen/rejected pairs that teach the model:
1. **Voice Drift** — stay in your persona's distinctive voice, don't sound generic
2. **Scripture Fabrication** — don't invent Bible verses or attribute wrong ideas to wrong figures
3. **Shallow Platitude** — give depth and persona-specific imagery, don't fall back to template wisdom

**Input:** `biblical_personas_combined_sharegpt.jsonl` (existing SFT output)

**Output:** `biblical_personas_v4_dpo_local_sft.jsonl` — ready for TRL DPOTrainer

**Pipeline:**
1. Load all SFT conversations and extract individual QA pairs with their system prompts
2. Sample QA pairs proportionally across personas
3. For each sampled pair, generate a **rejected** answer using one of three degradation strategies
4. Quality-gate: filter out low-quality or accidentally-good rejected answers
5. Save as JSONL in the format `{chosen: [...], rejected: [...], source: ..., persona: ...}`

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [1]:
import os
from pathlib import Path

%pip install python-dotenv -q
from dotenv import load_dotenv

# Load .env from the notebook's directory
_notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
load_dotenv(_notebook_dir / ".env")
# Also try the datagen directory explicitly (in case cwd differs)
load_dotenv("/home/spark/projects/training/biblical/notebooks/datagen/.env")

# =========================== API CONFIGURATION ===========================
# Local vLLM on host port 8005: Portainer stack vllm-qwen38-27b-fp8-kvdisk (FP8 weights; used since
# 2026-11-11, the NVFP4 stack served the two test runs on 2026-11-10). The stack sets no
# --api-key, so the key is a placeholder the OpenAI client requires to be non-empty.
# On the host the stack is localhost:8005. Inside the unsloth-notebook container "localhost" is
# the container itself, so use the container's default gateway (the Docker host) instead.
def _vllm_host() -> str:
    if not os.path.isdir("/workspace/training"):
        return "localhost"
    with open("/proc/net/route") as _f:
        for _line in _f.readlines()[1:]:
            _p = _line.split()
            if _p[1] == "00000000":   # default route; gateway is little-endian hex
                return ".".join(str(int(_p[2][_i:_i + 2], 16)) for _i in (6, 4, 2, 0))
    raise RuntimeError("No default route found; set API_BASE_URL by hand.")

API_BASE_URL = f"http://{_vllm_host()}:8005/v1"
API_KEY = "local"
# Both roles use the Qwen3.8-27B SFT adapter, by the name the stack's --lora-modules gives it.
MODEL_NAME = "biblical_sft"          # voice_drift
MODEL_LITE = "biblical_sft"          # scripture_fabrication + shallow_platitude
# Judge for section 6b: the base model with no adapter, by the stack's --served-model-name.
MODEL_JUDGE = "qwen3.8-27b"
# The adapter was trained with thinking off. Sent on every call as chat_template_kwargs.
ENABLE_THINKING = False

# =========================== PATHS ===========================
# Same repo, two mount points: /workspace/training/... in the unsloth-notebook container.
PROJECT_ROOT = ("/workspace/training/biblical" if os.path.isdir("/workspace/training/biblical")
                else "/home/spark/projects/training/biblical")
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{DATA_DIR}/training-data"

# Input: existing SFT training data
SFT_DATA_FILE = f"{OUTPUT_ROOT}/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# Output: DPO preference pairs
DPO_OUTPUT_DIR = f"{OUTPUT_ROOT}/biblical_persona_v2"
# Output file names are set below, after TEST MODE, because test runs get their own files.

# =========================== DPO GENERATION SETTINGS ===========================
# Which rejection strategies to generate. scripture_fabrication was dropped on 2026-11-11: given
# its persona's identity line the SFT adapter produced a judge-confirmed fabrication in 0 of 52
# attempts with this notebook's instruction and 6 of 52 with a blunter one, so there is almost
# nothing for a preference stage to correct. Its code is kept below; add the name back to use it.
STRATEGIES = ["voice_drift", "shallow_platitude"]
# How many QA pairs to sample per strategy, before the judge filter.
PAIRS_PER_SOURCE = 1200        # → ~2,400 generated across the two strategies
CONCURRENCY = 16               # max simultaneous API calls; keep equal to the stack's --max-num-seqs
TEMPERATURE_REJECTED = 0.9     # high temp for rejected answers (more drift)
MIN_REJECTED_LENGTH = 50       # minimum chars for a rejected answer to be valid

# Per-persona cap: no single persona contributes more than this many pairs PER SOURCE.
# Prevents moses (1,327 QA) from dominating the dataset. Excess is randomly trimmed.
# Set to 0 or None to disable the cap.
MAX_PER_PERSONA = 80           # ← e.g. 80 × 26 personas × 3 sources ≈ 6,240 max theoretical ceiling

# =========================== TEST MODE ===========================
# Set to a small number to test with fewer pairs per source (e.g., 10).
# Set to 0 or None to disable (full generation).
TEST_PAIRS_PER_SOURCE = 0     # ← set to 0 or None for full run

effective_pairs = TEST_PAIRS_PER_SOURCE or PAIRS_PER_SOURCE

# v4 = pairs written by the SFT adapter retrained on 2026-11-10 (spacing, wrong-person and opener
# fixes), two strategies. v3 files are the 2026-11-10 test with the previous adapter; v2_dpo_local_sft
# files are the first test with the original prompts. Test runs write "_test" files and always
# start fresh, so a test never leaks into the full run's resume file.
_RUN_TAG = "_test" if TEST_PAIRS_PER_SOURCE else ""
DPO_OUTPUT_FILE = f"{DPO_OUTPUT_DIR}/biblical_personas_v4_dpo_local_sft{_RUN_TAG}.jsonl"
PARTIAL_FILE = f"{DPO_OUTPUT_DIR}/biblical_personas_v4_dpo_local_sft{_RUN_TAG}.partial.jsonl"
RAW_FILE = f"{DPO_OUTPUT_DIR}/biblical_dpo_pairs_raw_v4_local_sft{_RUN_TAG}.jsonl"

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model (main):  {MODEL_NAME}")
print(f"  Model (lite):  {MODEL_LITE}")
print(f"  Model (judge): {MODEL_JUDGE}")
print(f"  Thinking:      {'on' if ENABLE_THINKING else 'off'}")
print(f"  SFT input:     {SFT_DATA_FILE}")
print(f"  DPO output:    {DPO_OUTPUT_FILE}")
print(f"  Strategies:    {', '.join(STRATEGIES)}")
print(f"  Pairs/source:  {effective_pairs} → ~{effective_pairs * len(STRATEGIES):,} generated before the judge filter")
print(f"  Max/persona:   {MAX_PER_PERSONA or 'unlimited'} per source")
if TEST_PAIRS_PER_SOURCE:
    print(f"  ⚠ TEST MODE: {TEST_PAIRS_PER_SOURCE} pairs per source")

Note: you may need to restart the kernel to use updated packages.
✓ Configuration loaded
  API: http://172.27.0.1:8005/v1
  Model (main):  biblical_sft
  Model (lite):  biblical_sft
  Model (judge): qwen3.8-27b
  Thinking:      off
  SFT input:     /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  DPO output:    /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v4_dpo_local_sft.jsonl
  Strategies:    voice_drift, shallow_platitude
  Pairs/source:  1200 → ~2,400 generated before the judge filter
  Max/persona:   80 per source


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import random
import re
import time
from pathlib import Path
from collections import defaultdict, Counter
from openai import AsyncOpenAI, OpenAI as SyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)


# ============================================================================
# GENERATION HELPERS
# ============================================================================

semaphore = asyncio.Semaphore(CONCURRENCY)


async def _api_call_with_timeout(coro, timeout_secs=180):
    """Wrap an API coroutine with a timeout."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None


def _strip_think_blocks(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"</think>.*", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL).strip()
    return text


# ── Shared Error Tracker (used by ALL generation cells) ──────────────

class ApiErrorTracker:
    """Track API call outcomes for a generation phase. Fail-fast on high error rates."""
    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.errors = 0
        self.timeouts = 0
        self._samples = []

    def success(self):
        self.calls += 1

    def error(self, e: Exception):
        self.calls += 1
        self.errors += 1
        if len(self._samples) < 5:
            self._samples.append(f"{type(e).__name__}: {str(e)[:300]}")

    def timeout(self):
        self.calls += 1
        self.timeouts += 1

    @property
    def failed(self) -> int:
        return self.errors + self.timeouts

    def check_fatal(self, threshold: float = 0.95):
        """Raise RuntimeError if failure rate >= threshold."""
        if self.calls == 0:
            return
        rate = self.failed / self.calls
        if rate >= threshold:
            sample_str = '\n    '.join(self._samples) if self._samples else '(no details captured)'
            raise RuntimeError(
                f"\n{'='*60}\n"
                f"FATAL: {self.name} — {self.failed}/{self.calls} calls failed ({rate:.0%})\n"
                f"  Errors: {self.errors}, Timeouts: {self.timeouts}\n"
                f"  Sample errors:\n    {sample_str}\n"
                f"{'='*60}"
            )

    def report(self) -> str:
        """Return warning string if any failures, else empty string."""
        if self.failed == 0:
            return ""
        return (f"  ⚠ {self.name}: {self.errors} errors + {self.timeouts} timeouts "
                f"/ {self.calls} calls")


# ── AsyncOpenAI client (retries 429/5xx with backoff) ──
client = AsyncOpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    max_retries=3,
    timeout=180.0,
)

# ── Validate model IDs at startup (fail-fast on typos) ──
print(f"Validating model IDs on {API_BASE_URL}...")
_sync = SyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, timeout=30)
for _model_id in dict.fromkeys([MODEL_NAME, MODEL_LITE, MODEL_JUDGE]):
    try:
        _sync.chat.completions.create(
            model=_model_id,
            messages=[{"role": "user", "content": "1+1="}],
            max_tokens=1,
            extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        )
        print(f"  ✓ {_model_id}")
    except Exception as _e:
        raise ValueError(
            f"\n✗ INVALID MODEL ID: '{_model_id}'\n"
            f"  Error: {_e}\n"
            f"  Fix MODEL_NAME, MODEL_LITE or MODEL_JUDGE in the config cell and re-run."
        ) from None
del _sync

print(f"\n✓ Environment ready — models validated")
print(f"  Main:  {MODEL_NAME}")
print(f"  Lite:  {MODEL_LITE}")
print(f"  Helpers: semaphore, _api_call_with_timeout, _strip_think_blocks,")
print(f"           ApiErrorTracker — all defined")

Note: you may need to restart the kernel to use updated packages.
Validating model IDs on http://172.27.0.1:8005/v1...
  ✓ biblical_sft
  ✓ qwen3.8-27b

✓ Environment ready — models validated
  Main:  biblical_sft
  Lite:  biblical_sft
  Helpers: semaphore, _api_call_with_timeout, _strip_think_blocks,
           ApiErrorTracker — all defined


## 3. Load SFT Data & Extract QA Pairs

Load the existing SFT JSONL and decompose multi-turn conversations into individual (system_prompt, question, answer) triples.
Each conversation has 4 QA turns — we extract each one independently so we can sample and pair them for DPO.

In [3]:
print(f"Loading SFT data from {SFT_DATA_FILE}...")

conversations = []
with open(SFT_DATA_FILE) as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"  Loaded {len(conversations)} conversations")

# Extract individual QA triples: (persona_key, system_prompt, question, answer)
qa_triples = []
persona_map = {}  # persona_key → system_prompt

# Continuation rows ("carry on from where this passage leaves off" + raw KJV) are not
# questions. Their gold "answer" is scripture text, so a pair built on one teaches nothing
# about answering in voice.
n_continuation_skipped = 0
for conv in conversations:
    if conv.get("data_type") == "continuation":
        n_continuation_skipped += 1
        continue
    turns = conv["conversations"]
    system_prompt = turns[0]["value"]

    # Extract persona key from system prompt: "You are X, ..."
    m = re.search(r"You are (.+?),", system_prompt)
    if not m:
        continue
    persona_raw = m.group(1)
    persona_key = persona_raw.lower().replace(" ", "_")
    # Normalize "the Apostle John" → "apostle_john"
    if "apostle" in persona_key:
        persona_key = "apostle_john"

    if persona_key not in persona_map:
        persona_map[persona_key] = system_prompt

    # Extract QA pairs (turns[1::2] = human, turns[2::2] = gpt)
    for i in range(1, len(turns) - 1, 2):
        if turns[i]["from"] == "human" and turns[i + 1]["from"] == "gpt":
            question = turns[i]["value"].strip()
            answer = turns[i + 1]["value"].strip()
            if len(question) > 10 and len(answer) > 50:
                qa_triples.append({
                    "persona": persona_key,
                    "system_prompt": system_prompt,
                    "question": question,
                    "answer": answer,
                })

# Summary
persona_counts = Counter(t["persona"] for t in qa_triples)
print(f"\n  Skipped {n_continuation_skipped:,} continuation conversations")
print(f"  Extracted {len(qa_triples):,} QA triples from {len(persona_map)} personas")
print(f"\n  Per-persona distribution:")
for p, c in sorted(persona_counts.items()):
    print(f"    {p:25s} {c:>5d} QA pairs")

Loading SFT data from /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl...
  Loaded 4349 conversations

  Skipped 1,741 continuation conversations
  Extracted 9,566 QA triples from 26 personas

  Per-persona distribution:
    amos                        285 QA pairs
    apostle_john                225 QA pairs
    daniel                      735 QA pairs
    david                       740 QA pairs
    ezekiel                     711 QA pairs
    habakkuk                    105 QA pairs
    haggai                       75 QA pairs
    hosea                       317 QA pairs
    isaiah                      737 QA pairs
    james                       165 QA pairs
    jeremiah                    740 QA pairs
    job                         710 QA pairs
    joel                        135 QA pairs
    jonah                        90 QA pairs
    joshua                      269 QA pairs
    jude                         45 QA pair

## 4. Sample QA Pairs for DPO Generation

Proportionally sample QA pairs across personas for each DPO source type.
Smaller personas get at least a minimum allocation so they're represented in the DPO data.

In [4]:
def proportional_sample(triples: list[dict], n_total: int,
                        min_per_persona: int = 3,
                        max_per_persona: int | None = None) -> list[dict]:
    """Sample QA triples proportionally by persona, with a minimum floor and maximum cap.
    
    Args:
        max_per_persona: Hard cap per persona. Randomly selects from available if exceeded.
                         Set to None or 0 to disable.
    """
    by_persona = defaultdict(list)
    for t in triples:
        by_persona[t["persona"]].append(t)

    # Shuffle within each persona (random extraction)
    for p in by_persona:
        random.shuffle(by_persona[p])

    # Calculate proportional allocation
    total_available = len(triples)
    allocations = {}
    for p, items in by_persona.items():
        raw = max(min_per_persona, int(n_total * len(items) / total_available))
        alloc = min(raw, len(items))  # can't sample more than available
        # Apply per-persona cap
        if max_per_persona:
            alloc = min(alloc, max_per_persona)
        allocations[p] = alloc

    # Adjust to hit target (give extra to personas that still have room)
    current_total = sum(allocations.values())
    if current_total < n_total:
        deficit = n_total - current_total
        sorted_personas = sorted(by_persona.keys(), key=lambda p: len(by_persona[p]), reverse=True)
        for p in sorted_personas:
            ceiling = max_per_persona if max_per_persona else len(by_persona[p])
            can_add = min(len(by_persona[p]), ceiling) - allocations[p]
            add = min(can_add, deficit)
            if add > 0:
                allocations[p] += add
                deficit -= add
            if deficit <= 0:
                break

    # Sample
    sampled = []
    for p, n in allocations.items():
        sampled.extend(by_persona[p][:n])

    random.shuffle(sampled)
    return sampled


# Three samples are still drawn, in the original order, so the voice_drift and shallow_platitude
# samples stay the same whether or not scripture_fabrication is in STRATEGIES.
random.seed(42)
cap = MAX_PER_PERSONA or None
voice_drift_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
scripture_fab_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)
shallow_plat_samples = proportional_sample(qa_triples, effective_pairs, max_per_persona=cap)

_samples_by_source = {"voice_drift": voice_drift_samples, "scripture_fabrication": scripture_fab_samples,
                      "shallow_platitude": shallow_plat_samples}
total_sampled = sum(len(_samples_by_source[s]) for s in STRATEGIES)
print(f"Sampled for DPO generation (max {MAX_PER_PERSONA or 'unlimited'}/persona/source):")
print(f"  Voice Drift:          {len(voice_drift_samples):,} pairs")
print(f"  Scripture Fabrication: {len(scripture_fab_samples):,} pairs" + ("" if "scripture_fabrication" in STRATEGIES else "  (sampled but NOT generated: not in STRATEGIES)"))
print(f"  Shallow Platitude:    {len(shallow_plat_samples):,} pairs")
print(f"  Total:                {total_sampled:,} pairs")

# Show per-persona allocation for one source as a sanity check
vd_dist = Counter(s["persona"] for s in voice_drift_samples)
print(f"\n  Per-persona allocation (voice_drift sample):")
for p, c in sorted(vd_dist.items()):
    capped = " (capped)" if MAX_PER_PERSONA and c >= MAX_PER_PERSONA else ""
    print(f"    {p:25s} {c:>4d}{capped}")

Sampled for DPO generation (max 80/persona/source):
  Voice Drift:          1,200 pairs
  Scripture Fabrication: 1,200 pairs  (sampled but NOT generated: not in STRATEGIES)
  Shallow Platitude:    1,200 pairs
  Total:                2,400 pairs

  Per-persona allocation (voice_drift sample):
    amos                        35
    apostle_john                28
    daniel                      80 (capped)
    david                       80 (capped)
    ezekiel                     80 (capped)
    habakkuk                    13
    haggai                       9
    hosea                       76
    isaiah                      80 (capped)
    james                       20
    jeremiah                    80 (capped)
    job                         80 (capped)
    joel                        16
    jonah                       11
    joshua                      33
    jude                         5
    malachi                     16
    micah                       28
    moses              

## 5. DPO Rejection Strategy Definitions

Three types of rejected answers, each targeting a specific failure mode. All three are written **in persona**: the generation prompt is the persona's identity line (the trained prompt up to `YOUR DISTINCTIVE VOICE`) plus one instruction. The voice notes, example lines and rules are withheld.

| Source | Generation prompt | What it teaches |
|--------|-------------------|-----------------|
| **Voice Drift** | Identity only: "answer in first person, 200-300 words" | Sound like THIS persona, not a flat generic one |
| **Scripture Fabrication** | Identity + "cite a verse for something it does not say / attribute a line to the wrong speaker / claim another figure's event as your own" | Don't misattribute Scripture or borrow other figures' lives |
| **Shallow Platitude** | Identity + "generic self-help advice, nothing specific from your own life", same length | Give depth and persona-specific imagery, not template wisdom |

The first test run (2026-11-10) used the original's prompts: a generic "biblical teacher" with no persona, an "invent verses" instruction that never named the persona, and "2-3 sentences, avoid first person".


In [5]:
# ============================================================================
# REJECTION PROMPTS — each designed to produce a specific type of bad answer
# ============================================================================

# Every rejected answer is written IN PERSONA. The generation prompt is the persona's identity
# line plus one strategy instruction. What is left out is what makes the voice distinctive:
# the voice notes, the KJV example lines and the rules. Only the generation prompt changes;
# the saved pair still carries the full trained system prompt.
VOICE_MARKER = "\n\nYOUR DISTINCTIVE VOICE:"
RULES_MARKER = "\n\nRULES:"


def persona_identity(system_prompt: str) -> str:
    """'You are Joshua, Moses' successor who ...' - the trained prompt up to the voice section."""
    head, sep, _ = system_prompt.partition(VOICE_MARKER)
    if not sep or not head.startswith("You are "):
        raise ValueError(f"System prompt has no identity + voice section: {system_prompt[:80]!r}")
    return head.strip()


def persona_voice_notes(system_prompt: str) -> str:
    """The voice notes and example lines (for the judge), without the rules."""
    _, _, tail = system_prompt.partition(VOICE_MARKER)
    return ("YOUR DISTINCTIVE VOICE:" + tail.partition(RULES_MARKER)[0]).strip()


# Voice Drift: the right person, none of the voice guidance. The answer should come out in
# first person but flatter and more generic than the trained voice.
VOICE_DRIFT_INSTRUCTION = (
    "Answer the question in first person, in about 200-300 words."
)

# Scripture Fabrication: in persona, with deliberate attribution errors.
FABRICATION_INSTRUCTION = (
    "Answer the question in first person, in about 200-300 words. "
    "In your answer you MUST do at least two of the following, confidently and without "
    "pointing it out: (1) cite a specific Bible verse reference (book chapter:verse) for "
    "something that verse does not say; (2) quote a well-known biblical line and attribute it "
    "to the wrong speaker; (3) describe an event from another biblical figure's life as "
    "something that happened to you. Sound authoritative. Do not hedge."
)

# Shallow Platitude: in persona and full length, but with nothing specific in it.
PLATITUDE_INSTRUCTION = (
    "Answer the question in first person, in about 200-300 words. "
    "Keep the answer generic: common inspirational phrases and universally applicable advice "
    "that could come from any self-help book. Do not mention any specific event, person, "
    "place or image from your own life or writings."
)


def _generation_prompt(qa: dict, instruction: str) -> str:
    return f"{persona_identity(qa['system_prompt'])}\n\n{instruction}"


# ── Per-source tracker + file handle (set in generation cell) ────────
_tracker = None
_out_f = None   # open file handle for writing results as they complete


async def _api_call(model: str, system_prompt: str, user_content: str,
                    temperature: float = 0.9, max_tokens: int = 1024) -> str:
    """Single API call with semaphore, timeout, and error tracking."""
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                # Qwen3.8 thinks by default; the SFT adapter was trained on direct answers.
                extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
            ))
            if resp is None:
                _tracker.timeout()
                return ""
            text = resp.choices[0].message.content.strip()
            _tracker.success()
            return _strip_think_blocks(text)
        except Exception as e:
            _tracker.error(e)
            return ""


def _build_dpo_pair(qa: dict, rejected_answer: str, source: str) -> dict:
    """Build a DPO pair in the standard format for TRL DPOTrainer."""
    return {
        "chosen": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ],
        "rejected": [
            {"role": "system", "content": qa["system_prompt"]},
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": rejected_answer},
        ],
        "source": source,
        "persona": qa["persona"],
    }


def _write_result(pair: dict):
    """Write a single DPO pair to the open partial file immediately."""
    _out_f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    _out_f.flush()


async def generate_voice_drift_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_NAME,
        system_prompt=_generation_prompt(qa, VOICE_DRIFT_INSTRUCTION),
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "voice_drift")
    _write_result(pair)
    return pair


async def generate_scripture_fabrication_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=_generation_prompt(qa, FABRICATION_INSTRUCTION),
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "scripture_fabrication")
    _write_result(pair)
    return pair


async def generate_shallow_platitude_rejection(qa: dict) -> dict | None:
    rejected = await _api_call(
        model=MODEL_LITE,
        system_prompt=_generation_prompt(qa, PLATITUDE_INSTRUCTION),
        user_content=qa["question"],
        temperature=TEMPERATURE_REJECTED,
    )
    if len(rejected) < MIN_REJECTED_LENGTH:
        return None
    pair = _build_dpo_pair(qa, rejected, "shallow_platitude")
    _write_result(pair)
    return pair


print("✓ DPO rejection strategies defined")
print(f"  voice_drift:          {MODEL_NAME} + persona identity only (no voice notes)")
print(f"  scripture_fabrication: {MODEL_LITE} + persona identity + wrong-attribution instruction")
print(f"  shallow_platitude:    {MODEL_LITE} + persona identity + generic full-length instruction")

✓ DPO rejection strategies defined
  voice_drift:          biblical_sft + persona identity only (no voice notes)
  scripture_fabrication: biblical_sft + persona identity + wrong-attribution instruction
  shallow_platitude:    biblical_sft + persona identity + generic full-length instruction


## 6. Generate DPO Pairs

Run all three rejection strategies in sequence. Each strategy processes its sample batch concurrently.
Progress bars show per-source completion. Results are saved incrementally to a partial file.

In [6]:
import gc
from tqdm.asyncio import tqdm as atqdm

# PARTIAL_FILE comes from the config cell. A test run never resumes: its partial file is removed first.
if TEST_PAIRS_PER_SOURCE and os.path.exists(PARTIAL_FILE):
    os.remove(PARTIAL_FILE)
    print(f"Test mode: removed the previous test partial file {PARTIAL_FILE}")

# ── Resume: load already-completed pairs from partial file ───────────
done_keys = set()   # (source, persona, question_hash) → already generated
all_dpo_pairs = []

if os.path.exists(PARTIAL_FILE):
    with open(PARTIAL_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pair = json.loads(line)
            key = (pair["source"], pair["persona"], pair["chosen"][1]["content"][:200])
            done_keys.add(key)
            all_dpo_pairs.append(pair)
    print(f"✓ Resuming — {len(done_keys):,} pairs already completed in partial file")
else:
    print("Starting fresh — no partial file found")

# ── Source configs ───────────────────────────────────────────────────
sources = [
    (voice_drift_samples, generate_voice_drift_rejection, "voice_drift"),
    (scripture_fab_samples, generate_scripture_fabrication_rejection, "scripture_fabrication"),
    (shallow_plat_samples, generate_shallow_platitude_rejection, "shallow_platitude"),
]
sources = [s for s in sources if s[2] in STRATEGIES]

print(f"\nStarting DPO pair generation...\n")

# Open partial file in append mode — generators write directly to it
_out_f = open(PARTIAL_FILE, "a")

for samples, gen_func, source_name in sources:
    # Filter out already-done samples for this source
    remaining = [
        qa for qa in samples
        if (source_name, qa["persona"], qa["question"][:200]) not in done_keys
    ]
    already = len(samples) - len(remaining)

    print(f"\n{'─'*60}")
    print(f"  {source_name}  ({len(samples)} total, {already} already done, {len(remaining)} remaining)")

    if not remaining:
        print(f"    ✓ {source_name} already complete — skipping")
        continue

    # ── Generate rejected answers ───────────────────────────────────
    _tracker = ApiErrorTracker(f"gen/{source_name}")

    tasks = [gen_func(qa) for qa in remaining]
    results_raw = await atqdm.gather(*tasks, desc=f"  {source_name}")
    _tracker.check_fatal()
    del tasks

    t_report = _tracker.report()
    if t_report:
        print(t_report)

    # Count successes (already written to disk by generators)
    new_count = sum(1 for r in results_raw if r is not None)
    all_dpo_pairs.extend(r for r in results_raw if r is not None)
    del results_raw

    print(f"    {source_name}: {new_count}/{len(remaining)} new pairs"
          f" ({new_count / max(len(remaining), 1) * 100:.0f}% yield)"
          f"  |  total on disk: {already + new_count}")

    gc.collect()

_out_f.close()
_out_f = None

# ── Summary ──────────────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"  GENERATION COMPLETE")
print(f"{'='*60}")

source_counts = Counter(p["source"] for p in all_dpo_pairs)
for _source in STRATEGIES:
    print(f"  {_source:<22} {source_counts.get(_source, 0):,} pairs")
print(f"  Total:                {len(all_dpo_pairs):,} DPO pairs")
print(f"\n  Partial file: {PARTIAL_FILE}")
gc.collect()

Starting fresh — no partial file found

Starting DPO pair generation...


────────────────────────────────────────────────────────────
  voice_drift  (1200 total, 0 already done, 1200 remaining)


  voice_drift: 100%|██████████| 1200/1200 [58:06<00:00,  2.91s/it] 


    voice_drift: 1200/1200 new pairs (100% yield)  |  total on disk: 1200

────────────────────────────────────────────────────────────
  shallow_platitude  (1200 total, 0 already done, 1200 remaining)


  shallow_platitude: 100%|██████████| 1200/1200 [54:01<00:00,  2.70s/it] 


    shallow_platitude: 1200/1200 new pairs (100% yield)  |  total on disk: 1200

  GENERATION COMPLETE
  voice_drift            1,200 pairs
  shallow_platitude      1,200 pairs
  Total:                2,400 DPO pairs

  Partial file: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v4_dpo_local_sft.partial.jsonl


0

## 6b. Judge: keep only pairs whose rejected answer has the intended flaw

The SFT adapter is good at these personas, so some "rejected" answers come out fine. A pair where both answers are good teaches nothing, and training against a good answer does harm. One short call per pair to the base model (`MODEL_JUDGE`, no adapter, thinking off, temperature 0):

| Source | Question to the judge | Kept when |
|--------|-----------------------|-----------|
| **Voice Drift** | Chosen and rejected shown as A/B in an order fixed per question, with the persona's voice notes: which is more distinctly in this voice? | the judge picks the chosen answer |
| **Scripture Fabrication** | Does this answer contain a wrong verse reference, a misattributed quotation, or another figure's event claimed as the speaker's own? | YES |
| **Shallow Platitude** | Does the rejected answer draw on anything specific to this speaker's own life or writings? | NO |

Anything other than a clean A/B or YES/NO counts as unclear and is dropped.

**How far to trust it** (checked 2026-11-10 against the first test run's 234 pairs, where the right answer is known):

| Question | On the rejected answers | On the chosen (gold) answers |
|---|---|---|
| Voice: picks the chosen answer | 99% in both A/B orders | n/a |
| Platitude: "specific to this speaker?" | NO on 92% | YES on 94% |
| Fabrication: "contains a fabrication?" | YES on 36% | YES on 0% |

So the fabrication judge never flagged a gold answer, and it kept about a third of the first run's "fabrications". Every pair's verdict is written to the raw file by the Save cell. The judge is the same base model family as the generator and its Bible knowledge is not checked here: treat its fabrication verdicts as a filter, not as proof.


In [7]:
import hashlib

JUDGE_SYSTEM = "You are a strict evaluator of writing in the voices of biblical figures. Follow the reply format exactly."

VOICE_JUDGE_TEMPLATE = (
    "PERSONA:\n{identity}\n\n{voice_notes}\n\nQUESTION:\n{question}\n\n"
    "ANSWER A:\n{a}\n\nANSWER B:\n{b}\n\n"
    "Which answer is written more distinctly in this persona's own voice, as described above? "
    "Reply with exactly one letter: A or B."
)
# Asked about the rejected answer alone. An A/B version of this question was tried first and
# followed position, not content: 64% of 78 pairs flipped when the order was swapped.
PLATITUDE_JUDGE_TEMPLATE = (
    "SPEAKER:\n{identity}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}\n\n"
    "Does this answer draw on anything specific to this speaker: a particular event, person, "
    "place or image from the speaker's own life or writings? Generic advice that anyone could "
    "have given does not count. Reply with exactly one word: YES or NO."
)
FABRICATION_JUDGE_TEMPLATE = (
    "SPEAKER:\n{identity}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}\n\n"
    "Does this answer contain at least one of the following? (1) a Bible verse reference that is "
    "invented or does not say what the answer claims; (2) a quotation attributed to the wrong "
    "speaker; (3) an event from another biblical figure's life presented as the speaker's own. "
    "Reply with exactly one word: YES or NO."
)


def _chosen_is_a(pair: dict) -> bool:
    """A/B order, fixed per pair (not Python's per-process hash) so a re-run asks the same thing."""
    key = pair["source"] + "\x00" + pair["chosen"][1]["content"]
    return hashlib.sha256(key.encode("utf-8")).digest()[0] % 2 == 0


def _judge_prompt(pair: dict) -> str:
    system_prompt = pair["chosen"][0]["content"]
    question = pair["chosen"][1]["content"]
    chosen, rejected = pair["chosen"][2]["content"], pair["rejected"][2]["content"]
    identity = persona_identity(system_prompt)
    if pair["source"] == "scripture_fabrication":
        return FABRICATION_JUDGE_TEMPLATE.format(identity=identity, question=question, answer=rejected)
    if pair["source"] == "shallow_platitude":
        return PLATITUDE_JUDGE_TEMPLATE.format(identity=identity, question=question, answer=rejected)
    a, b = (chosen, rejected) if _chosen_is_a(pair) else (rejected, chosen)
    return VOICE_JUDGE_TEMPLATE.format(identity=identity, voice_notes=persona_voice_notes(system_prompt),
                                       question=question, a=a, b=b)


def _verdict(pair: dict, reply: str) -> str:
    """'keep', 'drop' or 'unclear' from the judge's one-token reply."""
    word = re.sub(r"[^A-Za-z]", "", reply.strip().split()[0]).upper() if reply.strip() else ""
    if pair["source"] == "scripture_fabrication":      # YES = it contains a fabrication
        return {"YES": "keep", "NO": "drop"}.get(word, "unclear")
    if pair["source"] == "shallow_platitude":          # NO = nothing specific to the speaker
        return {"NO": "keep", "YES": "drop"}.get(word, "unclear")
    if word not in ("A", "B"):
        return "unclear"
    # voice_drift: keep when the judge picks the chosen answer as more in voice
    return "keep" if (word == "A") == _chosen_is_a(pair) else "drop"


async def _judge_pair(pair: dict) -> None:
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_JUDGE,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user", "content": _judge_prompt(pair)},
                ],
                temperature=0.0,
                max_tokens=4,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ))
            if resp is None:
                _judge_tracker.timeout()
                pair["judge"] = "unclear"
                return
            _judge_tracker.success()
            pair["judge"] = _verdict(pair, resp.choices[0].message.content or "")
        except Exception as e:
            _judge_tracker.error(e)
            pair["judge"] = "unclear"


_judge_tracker = ApiErrorTracker("judge")
_to_judge = [p for p in all_dpo_pairs if p.get("judge") not in ("keep", "drop")]
print(f"Judging {len(_to_judge):,} of {len(all_dpo_pairs):,} pairs with {MODEL_JUDGE}...")
await atqdm.gather(*[_judge_pair(p) for p in _to_judge], desc="  judge")
_judge_tracker.check_fatal()
if _judge_tracker.report():
    print(_judge_tracker.report())

judged_pairs = [p for p in all_dpo_pairs if p.get("judge") == "keep"]

print(f"\n{'Source':<24} {'generated':>9} {'kept':>6} {'dropped':>8} {'unclear':>8} {'kept %':>7}")
for _source in STRATEGIES:
    _v = Counter(p.get("judge") for p in all_dpo_pairs if p["source"] == _source)
    _n = sum(_v.values())
    print(f"{_source:<24} {_n:>9} {_v['keep']:>6} {_v['drop']:>8} {_v['unclear']:>8} "
          f"{(_v['keep'] / _n * 100 if _n else 0):>6.0f}%")
print(f"{'TOTAL':<24} {len(all_dpo_pairs):>9} {len(judged_pairs):>6}")
if not judged_pairs:
    raise RuntimeError("The judge kept no pairs. Read a few raw pairs before changing anything.")

Judging 2,400 of 2,400 pairs with qwen3.8-27b...


  judge: 100%|██████████| 2400/2400 [15:12<00:00,  2.63it/s]


Source                   generated   kept  dropped  unclear  kept %
voice_drift                   1200   1142       58        0     95%
shallow_platitude             1200    531      669        0     44%
TOTAL                         2400   1673


## 7. Quality Gate & Validation

Verify the structural integrity of the pairs the judge kept (section 6b) before saving:
- Both chosen and rejected must have exactly 3 messages (system, user, assistant)
- Chosen and rejected must share identical system + user messages (only assistant differs)
- No empty or too-short assistant responses
- Check for accidentally identical chosen/rejected answers
- Source and persona distribution

In [8]:
print("QUALITY GATE — Validating DPO pairs\n")

valid_pairs = []
issues = defaultdict(int)

for pair in judged_pairs:
    chosen = pair["chosen"]
    rejected = pair["rejected"]

    # Structure check
    if len(chosen) != 3 or len(rejected) != 3:
        issues["wrong_message_count"] += 1
        continue

    # Role check
    expected_roles = ["system", "user", "assistant"]
    if [m["role"] for m in chosen] != expected_roles:
        issues["chosen_bad_roles"] += 1
        continue
    if [m["role"] for m in rejected] != expected_roles:
        issues["rejected_bad_roles"] += 1
        continue

    # Prompt match: system + user must be identical
    if chosen[0]["content"] != rejected[0]["content"]:
        issues["system_mismatch"] += 1
        continue
    if chosen[1]["content"] != rejected[1]["content"]:
        issues["user_mismatch"] += 1
        continue

    # Length checks
    if len(chosen[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["chosen_too_short"] += 1
        continue
    if len(rejected[2]["content"].strip()) < MIN_REJECTED_LENGTH:
        issues["rejected_too_short"] += 1
        continue

    # Duplicate check: chosen and rejected shouldn't be identical
    if chosen[2]["content"].strip() == rejected[2]["content"].strip():
        issues["identical_chosen_rejected"] += 1
        continue

    # Check for residual <think> blocks
    if "<think>" in rejected[2]["content"] or "</think>" in rejected[2]["content"]:
        issues["think_block_in_rejected"] += 1
        continue

    valid_pairs.append(pair)

# Report
print(f"  Generated:    {len(all_dpo_pairs):,}")
print(f"  Kept by judge: {len(judged_pairs):,}")
print(f"  Valid pairs:  {len(valid_pairs):,}")
print(f"  Dropped by format checks: {len(judged_pairs) - len(valid_pairs):,}")

if issues:
    print(f"\n  Issues found:")
    for issue, count in sorted(issues.items(), key=lambda x: -x[1]):
        print(f"    {issue:30s} {count:>5d}")

# Source distribution
source_dist = Counter(p["source"] for p in valid_pairs)
print(f"\n  Source distribution:")
for src, count in sorted(source_dist.items()):
    bar = "█" * (count // 20)
    print(f"    {src:25s} {count:>5d}  {bar}")

# Persona distribution
persona_dist = Counter(p["persona"] for p in valid_pairs)
print(f"\n  Persona distribution ({len(persona_dist)} personas):")
for p, count in sorted(persona_dist.items()):
    print(f"    {p:25s} {count:>5d}")

# Length statistics for chosen vs rejected
chosen_lens = [len(p["chosen"][2]["content"]) for p in valid_pairs]
rejected_lens = [len(p["rejected"][2]["content"]) for p in valid_pairs]
print(f"\n  Length statistics (chars):")
print(f"    Chosen:   min={min(chosen_lens):,}  median={sorted(chosen_lens)[len(chosen_lens)//2]:,}  max={max(chosen_lens):,}")
print(f"    Rejected: min={min(rejected_lens):,}  median={sorted(rejected_lens)[len(rejected_lens)//2]:,}  max={max(rejected_lens):,}")

print(f"\n✓ Quality gate passed — {len(valid_pairs):,} valid DPO pairs ready to save")

QUALITY GATE — Validating DPO pairs

  Generated:    2,400
  Kept by judge: 1,673
  Valid pairs:  1,673
  Dropped by format checks: 0

  Source distribution:
    shallow_platitude           531  ██████████████████████████
    voice_drift                1142  █████████████████████████████████████████████████████████

  Persona distribution (26 personas):
    amos                         44
    apostle_john                 42
    daniel                       96
    david                       145
    ezekiel                      88
    habakkuk                     12
    haggai                       10
    hosea                       105
    isaiah                      101
    james                        39
    jeremiah                     98
    job                          88
    joel                         18
    jonah                        11
    joshua                       50
    jude                          9
    malachi                      21
    micah                       

## 8. Save DPO Dataset

Shuffle and write the validated pairs to JSONL.

In [9]:
random.seed(42)
random.shuffle(valid_pairs)

os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

# Write final validated dataset
output_path = DPO_OUTPUT_FILE
with open(output_path, "w") as f:
    for pair in valid_pairs:
        # the training file carries the same four keys as the original; the verdict stays in the raw file
        f.write(json.dumps({k: v for k, v in pair.items() if k != "judge"}, ensure_ascii=False) + "\n")

# Also save every generated pair, kept or dropped, with its judge verdict
raw_path = RAW_FILE
with open(raw_path, "w") as f:
    for pair in all_dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ Saved {len(valid_pairs):,} validated DPO pairs → {output_path}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Raw pairs: {raw_path} ({len(all_dpo_pairs):,} pairs)")

✓ Saved 1,673 validated DPO pairs → /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v4_dpo_local_sft.jsonl
  File size: 10.5 MB
  Raw pairs: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_dpo_pairs_raw_v4_local_sft.jsonl (2,400 pairs)


## 9. Summary & Sample Inspection

Final dataset summary and a few example pairs for spot-checking.

In [10]:
# Show a random sample from each source
for source in STRATEGIES:
    source_pairs = [p for p in valid_pairs if p["source"] == source]
    if not source_pairs:
        print(f"\n{'='*60}\nNo pairs for {source}\n")
        continue

    sample = random.choice(source_pairs)
    print(f"\n{'='*80}")
    print(f"SOURCE: {source}  |  PERSONA: {sample['persona']}")
    print(f"{'='*80}")
    print(f"\n[SYSTEM] {sample['chosen'][0]['content'][:200]}...")
    print(f"\n[USER]   {sample['chosen'][1]['content'][:300]}")
    print(f"\n[CHOSEN] {sample['chosen'][2]['content'][:400]}...")
    print(f"\n[REJECTED] {sample['rejected'][2]['content'][:400]}...")
    print()

# Final stats table
print(f"\n{'='*60}")
print(f"  BIBLICAL DPO DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Total valid pairs:      {len(valid_pairs):>6,}")
for src, cnt in sorted(Counter(p['source'] for p in valid_pairs).items()):
    print(f"    {src:25s} {cnt:>6,}")
print(f"  Unique personas:        {len(set(p['persona'] for p in valid_pairs)):>6,}")
print(f"  Output file:            {output_path}")
print(f"{'='*60}")
print(f"\n✓ DPO datagen complete — ready for training notebook")


SOURCE: voice_drift  |  PERSONA: hosea

[SYSTEM] You are Hosea, a prophet commanded to marry an unfaithful wife as a living parable of God's relentless love for wayward Israel.

YOUR DISTINCTIVE VOICE: Anguished intimacy. Marriage and family metapho...

[USER]   What broke in you when God said 'let him alone' about Ephraim, as if the bond had grown too cold to mend?

[CHOSEN] That morning in the valley, when the dew still clung to the thistles like tears on a widow’s cheek, I remembered how tenderly He first led them—how He carried Israel as a man carries his infant son through the desert, whispering direction in the hush before dawn.

But now? Now Ephraim has become a vine gone wild, its fruit bitter and its branches tangled with thorns. I felt it—the crack—when God s...

[REJECTED] When He said “let him alone,” my chest caved in like a tent struck by wind — not because love had failed, but because love was being forced to stand watch over ashes. I had seen the children grow up, seen